In [2]:
# ============================================================
# K-MEANS CLUSTERING
# Elbow Method + Silhouette Score
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# ------------------------------------------------------------
# 1. Load Dataset
# ------------------------------------------------------------

df = pd.read_csv("/content/placement_predict_50k_adjusted (1).csv")

print("Dataset Shape:", df.shape)
display(df.head())

# ------------------------------------------------------------
# 2. Select Features
# ------------------------------------------------------------

numeric_cols = [
    "CGPA",
    "AttendancePercent",
    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",
    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore"
]

categorical_cols = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "HistoryOfBacklogs",
    "ExtraCurricular"
]

# ------------------------------------------------------------
# 3. Handle Missing Values
# ------------------------------------------------------------

imputer = SimpleImputer(strategy="median")

num_data = pd.DataFrame(
    imputer.fit_transform(df[numeric_cols]),
    columns=numeric_cols
)

# ------------------------------------------------------------
# 4. Convert Categorical Data to Numerical
# ------------------------------------------------------------

cat_data = pd.get_dummies(
    df[categorical_cols],
    drop_first=True
)

features = pd.concat(
    [num_data, cat_data],
    axis=1
)

# ------------------------------------------------------------
# 5. Standardization
# ------------------------------------------------------------

scaler = StandardScaler()

X = scaler.fit_transform(features)

print("Number of Features:", X.shape[1])

# ------------------------------------------------------------
# 6. Elbow Method
# ------------------------------------------------------------

inertia = []

K = range(2, 9)

for k in K:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model.fit(X)

    inertia.append(model.inertia_)

plt.figure(figsize=(7,5))

plt.plot(
    K,
    inertia,
    marker="o"
)

plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.title("K-Means Elbow Method")
plt.grid(True)
plt.show()

# ------------------------------------------------------------
# 7. Silhouette Score
# ------------------------------------------------------------

silhouette_scores = []

for k in K:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(X)

    score = silhouette_score(
        X,
        labels,
        sample_size=min(5000, len(X)),
        random_state=42
    )

    silhouette_scores.append(score)

plt.figure(figsize=(7,5))

plt.plot(
    K,
    silhouette_scores,
    marker="o"
)

plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")
plt.title("K-Means Silhouette Score")
plt.grid(True)
plt.show()

# ------------------------------------------------------------
# 8. Select Best K
# ------------------------------------------------------------

best_k = K[np.argmax(silhouette_scores)]

print("Best Number of Clusters:", best_k)
print("Best Silhouette Score:",
      max(silhouette_scores))

# ------------------------------------------------------------
# 9. Final K-Means Model
# ------------------------------------------------------------

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(X)

df["KMeans_Cluster"] = clusters

print("\nCluster Counts:")
print(df["KMeans_Cluster"].value_counts().sort_index())

# ------------------------------------------------------------
# 10. PCA Visualization
# ------------------------------------------------------------

pca = PCA(n_components=2)

X_pca = pca.fit_transform(X)

plt.figure(figsize=(8,6))

plt.scatter(
    X_pca[:,0],
    X_pca[:,1],
    c=clusters,
    cmap="viridis",
    s=10
)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("K-Means Clustering")
plt.colorbar(label="Cluster")

plt.show()

Dataset Shape: (50000, 21)


,Gender,City,CollegeTier,Stream,Specialisation,Hostel,HistoryOfBacklogs,CGPA,AttendancePercent,Internships,...,Workshops,Certifications,Publications,AptitudeTestScore,SoftSkillsRating,CodingTestScore,MockInterviewScore,ExtraCurricular,PlacementStatus,IsAnomaly
0,Female,Delhi,Tier3,IT,DataScience,Yes,Yes,6.63,68.3,2,...,0.0,1,0,62.3,6.57,40.6,65.7,No,0,0
1,Male,Chennai,Tier2,ECE,AI,Yes,No,6.40,71.0,1,...,0.0,2,0,44.0,5.86,40.3,51.8,No,0,0
2,Female,Hyderabad,Tier3,ECE,Networking,No,No,7.73,75.1,1,...,2.0,2,1,73.8,7.50,73.6,67.9,No,1,0
3,Female,Jaipur,Tier3,ECE,Embedded,No,No,9.73,99.2,4,...,5.0,6,2,100.0,9.41,98.7,NaN,No,1,0
4,Male,Ahmedabad,Tier3,Mechanical,DataScience,No,No,9.01,99.6,2,...,NaN,4,2,90.8,9.24,83.1,100.0,No,1,0


Number of Features: 33
Best Number of Clusters: 2
Best Silhouette Score: 0.14371388521428713

Cluster Counts:
KMeans_Cluster
0    26429
1    23571
Name: count, dtype: int64
